# Enzyme Function Prediction Using Graph Neural Networks and Protein Structure
This notebook is a toy project for learning Graph Neural Networks and applying them to protein structure. More precisely, given the structure of an enzyme this model will predict if it is an oxyreductase or a transferase.

##1. Data Manipulation

The protein structures were downloaded from RCSB PDB in format mmCIF. We selected proteins from Escherichia coli with structure experimentally determined by X-ray diffraction with a refinement resolution of at most 2.5Å having Enzyme Comission number exclusively 1 or exclusively 2.

###1.1 Obtaining $\alpha$-carbons scaffold of the structure
The $C_{\alpha}$ of an amino acid can be viewed as its "central" point, keeping only these carbons in the structure of the protein can be seen as a simplification that retains the essential information about the molecule structure.

In [ ]:
from Bio.PDB import MMCIFParser

#This function return the list of the spatial coordinates of the alpha-carbons
#of a given mmCIF file. It uses a mmCIF parser from Biopython.
def get_alpha_carbons(cif_file):
  parser = MMCIFParser(QUIET=True)
  structure = parser.get_structure("protein", cif_file)

  ca_coords = []

  for model in structure:
    for chain in model:
      for residue in chain:
        if "CA" in residue:
          atom = residue["CA"]
          coord = atom.get_coord()
          aa = residue.resname
          ca_coords.append((aa,coord))

  return ca_coords

In [ ]:
import os

#(Truncated) output from the above defined function
get_alpha_carbons('data/EC_01/1aa6.cif')[:5]

[('MET', array([68.592,  3.44 , 30.52 ], dtype=float32)),
 ('LYS', array([71.858,  4.86 , 31.889], dtype=float32)),
 ('LYS', array([73.133,  8.159, 33.285], dtype=float32)),
 ('VAL', array([76.264,  9.713, 31.806], dtype=float32)),
 ('VAL', array([77.941, 12.384, 33.981], dtype=float32))]

###1.2 Creating graph representation

We now create a graph that represents the protein structure, the nodes will be $C_{\alpha}$ and there will be an edge betwen two carbons when their distance is less than a defined distance cutoff. The graph will be represented using edge index.

In [ ]:
import numpy as np

def generate_graph(ca_list, dist_cutoff=8):

  edge_index_start = []
  edge_index_end = []

  N = len(ca_list)

  for i in range(N):
    for j in range(i,N):
      if np.linalg.norm(ca_list[i][1]-ca_list[j][1]) < dist_cutoff:
        edge_index_start.append(i)
        edge_index_end.append(j)

        if i != j:
          edge_index_start.append(j)
          edge_index_end.append(i)

  edge_index = np.array([edge_index_start, edge_index_end])
  return edge_index

In [40]:
#(Truncated) example of an edge list
ca_list = get_alpha_carbons('data/EC_01/1aa6.cif')
a, b = generate_graph(ca_list)

for i,j in zip(a,b):
  print(i,j)

  if i > 5:
    break

0 0
0 1
1 0
0 2
2 0
0 19
19 0


###1.3 Batch processing of all structures

In [53]:
from tqdm import tqdm

data_dirs = ['data/EC_01', 'data/EC_02']
all_alpha_carbons_01 = []
all_alpha_carbons_02 = []

for data_dir in data_dirs:
  print(f"Processing files in {data_dir}")
  for filename in tqdm(os.listdir(data_dir)):
    file_path = os.path.join(data_dir, filename)
    ca_coords = get_alpha_carbons(file_path)
    pdb_id = file_path[-8:-4]
    if data_dir == 'data/EC_01':
      all_alpha_carbons_01.append((pdb_id, ca_coords))
    elif data_dir == 'data/EC_02':
      all_alpha_carbons_02.append((pdb_id, ca_coords))

Processing files in data/EC_01


100%|██████████| 471/471 [03:45<00:00,  2.09it/s]


Processing files in data/EC_02


100%|██████████| 752/752 [05:18<00:00,  2.36it/s]


In [54]:
import pickle

output_dir = 'data_processed'
os.makedirs(output_dir, exist_ok=True)

output_path_01 = os.path.join(output_dir, 'all_alpha_carbons_01.pkl')
with open(output_path_01, 'wb') as f:
    pickle.dump(all_alpha_carbons_01, f)
print(f"Saved all_alpha_carbons_01 to {output_path_01}")

output_path_02 = os.path.join(output_dir, 'all_alpha_carbons_02.pkl')
with open(output_path_02, 'wb') as f:
    pickle.dump(all_alpha_carbons_02, f)
print(f"Saved all_alpha_carbons_02 to {output_path_02}")

Saved all_alpha_carbons_01 to data_processed/all_alpha_carbons_01.pkl
Saved all_alpha_carbons_02 to data_processed/all_alpha_carbons_02.pkl
